<a href="https://colab.research.google.com/github/alokrajcs24/AI-GC-LAB/blob/main/PropositionalLOGIC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import ast
import itertools
import re


class PropositionalModelChecker:

    @staticmethod
    def _preprocess_expr(expr: str) -> str:
        """Converts logical operator symbols into standard Python word/symbol operators."""
        # Normalize whitespace
        expr = expr.strip()

        # Handle operators with proper boundary matching
        expr = expr.replace("<->", " == ")  # Biconditional: P <-> Q is P == Q
        expr = expr.replace("->", " <= ")  # Implication: P -> Q is equivalent to P <= Q in Boolean logic (False <= True is True, True <= False is False)
        expr = expr.replace("~", " not ")

        expr = re.sub(r"\bNOT\b", "not ", expr, flags=re.IGNORECASE)
        expr = re.sub(r"\bAND\b", " and ", expr, flags=re.IGNORECASE)
        expr = re.sub(r"\bOR\b", " or ", expr, flags=re.IGNORECASE)
        expr = re.sub(r"\bIMPLIES\b", " <= ", expr, flags=re.IGNORECASE)
        expr = re.sub(r"\bBICONDITIONAL\b", " == ", expr, flags=re.IGNORECASE)

        return expr

    @classmethod
    def evaluate_expression(cls, expr: str, assignment: dict) -> bool:
        """Evaluates a logical expression safely given a dictionary of variable truth assignments."""
        processed_expr = cls._preprocess_expr(expr)

        # Substitute variable names with their assigned Boolean values
        for var in sorted(assignment.keys(), key=len, reverse=True):
            val_str = str(assignment[var])
            processed_expr = re.sub(rf"\b{var}\b", val_str, processed_expr)

        # Safely evaluate using ast.literal_eval logic for pure boolean expressions
        parsed_ast = ast.parse(processed_expr, mode="eval")
        return bool(eval(compile(parsed_ast, filename="<ast>", mode="eval")))

    @classmethod
    def generate_truth_table(cls, expr: str, variables: list):
        """Generates and displays the complete truth table for the given expression."""
        print(f"Expression: {expr}\n")

        # Format header
        header_vars = " | ".join(f"{v:^5}" for v in variables)
        header = f"{header_vars} | Result "
        print(header)
        print("-" * len(header))

        models = []
        combinations = list(itertools.product([True, False], repeat=len(variables)))

        for combo in combinations:
            assignment = dict(zip(variables, combo))
            result = cls.evaluate_expression(expr, assignment)

            row_vars = " | ".join(f"{str(val):^5}" for val in combo)
            print(f"{row_vars} | {str(result):^6}")

            if result:
                models.append(assignment)

        print("\nSatisfiable Models:")
        for m in models:
            print(m)

        if len(models) == len(combinations):
            print("\nClassification: TAUTOLOGY (Valid)")
        elif len(models) == 0:
            print("\nClassification: CONTRADICTION (Unsatisfiable)")
        else:
            print("\nClassification: CONTINGENT (Satisfiable)")


# --- Example Execution ---
if __name__ == "__main__":
    checker = PropositionalModelChecker()

    # 1. Equivalency Test
    variables1 = ["P", "Q"]
    expr1 = "(P -> Q) <-> (~P OR Q)"
    print("--- Example 1: Equivalency Test ---")
    checker.generate_truth_table(expr1, variables1)

    print("\n" + "=" * 40 + "\n")

    # 2. 3-Variable Model Check
    variables2 = ["P", "Q", "R"]
    expr2 = "(P AND Q) -> (R OR NOT Q)"
    print("--- Example 2: 3-Variable Model Check ---")
    checker.generate_truth_table(expr2, variables2)

--- Example 1: Equivalency Test ---
Expression: (P -> Q) <-> (~P OR Q)

  P   |   Q   | Result 
-----------------------
True  | True  |  True 
True  | False |  True 
False | True  |  True 
False | False |  True 

Satisfiable Models:
{'P': True, 'Q': True}
{'P': True, 'Q': False}
{'P': False, 'Q': True}
{'P': False, 'Q': False}

Classification: TAUTOLOGY (Valid)


--- Example 2: 3-Variable Model Check ---
Expression: (P AND Q) -> (R OR NOT Q)

  P   |   Q   |   R   | Result 
-------------------------------
True  | True  | True  |  True 
True  | True  | False | False 
True  | False | True  |  True 
True  | False | False |  True 
False | True  | True  |  True 
False | True  | False |  True 
False | False | True  |  True 
False | False | False |  True 

Satisfiable Models:
{'P': True, 'Q': True, 'R': True}
{'P': True, 'Q': False, 'R': True}
{'P': True, 'Q': False, 'R': False}
{'P': False, 'Q': True, 'R': True}
{'P': False, 'Q': True, 'R': False}
{'P': False, 'Q': False, 'R': True}
{'P': Fa